<a href="https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TahaShan16/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
%pip -q install duckdb huggingface_hub scikit-learn

In [14]:
import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is missing. Add it in Colab Secrets first."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

TABLES = {
    "fact_daily": "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')",
}

MONTH = "2026-03"

print("Connected to FlyRank warehouse")

Connected to FlyRank warehouse


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline will rank pages for refresh review using two signals: staleness and CTR weakness. Staleness is linked to FlyRank refresh-style flags, because old content may need review. CTR weakness is linked to CTR-fix logic, because pages with visibility but low click-through may have title, snippet, or intent mismatch issues. I will check both signals before using them in the rule.

In [15]:
daily_columns = con.sql(f"""
    SELECT *
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
    LIMIT 1
""").df().columns.tolist()

daily_columns

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [16]:
content_columns = con.sql("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    LIMIT 1
""").df().columns.tolist()

content_columns

['client_hash_id',
 'content_hash_id',
 'keyword_hash_id',
 'url_hash_id',
 'keyword_char_count',
 'keyword_token_count',
 'url_char_count',
 'content_created_date',
 'content_updated_date',
 'content_type',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'main_intent',
 'backlinks',
 'category_count',
 'keyword_created_date',
 'provider_used',
 'model_used',
 'char_count',
 'word_count',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published',
 'is_deleted']

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal check 1: staleness bucket vs next-month impression decline
signal_df = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_april
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-04'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    content AS (
        SELECT
            client_hash_id,
            content_hash_id,
            content_type,
            word_count,
            COALESCE(last_optimized_date, content_updated_date, content_created_date) AS last_content_touch
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
        WHERE is_published IS TRUE
          AND is_deleted IS FALSE
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        c.content_type,
        c.word_count,
        DATE_DIFF('day', c.last_content_touch, DATE '2026-03-31') AS days_since_last_touch,
        m.impressions_march,
        m.clicks_march,
        100.0 * m.clicks_march / NULLIF(m.impressions_march, 0) AS ctr_march,
        m.avg_position_march,
        a.impressions_april,
        CASE
            WHEN a.impressions_april < 0.8 * m.impressions_march THEN 1
            ELSE 0
        END AS declined_next_month
    FROM march m
    INNER JOIN april a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
    INNER JOIN content c
      ON m.client_hash_id = c.client_hash_id
     AND m.content_hash_id = c.content_hash_id
""").df()

signal_df["staleness_bucket"] = pd.cut(
    signal_df["days_since_last_touch"],
    bins=[-1, 30, 90, 180, 365, 10_000],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_audit = (
    signal_df
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        decline_rate=("declined_next_month", "mean"),
        median_impressions=("impressions_march", "median")
    )
    .reset_index()
)

print("Signal 1 verdict: MIXED")
print("n:", len(signal_df))
staleness_audit

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 verdict: MIXED
n: 100874


,staleness_bucket,n,decline_rate,median_impressions
0,0-30,91,0.692308,441.0
1,31-90,17703,0.603909,713.0
2,91-180,122,0.549180,283.0
3,181-365,15,0.933333,482.0


In [18]:
# Signal check 2: CTR bucket vs next-month impression decline
ctr_signal_df = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_april
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-04'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_march,
        m.clicks_march,
        100.0 * m.clicks_march / NULLIF(m.impressions_march, 0) AS ctr_march,
        m.avg_position_march,
        a.impressions_april,
        CASE
            WHEN a.impressions_april < 0.8 * m.impressions_march THEN 1
            ELSE 0
        END AS declined_next_month
    FROM march m
    INNER JOIN april a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
""").df()

ctr_signal_df = ctr_signal_df[ctr_signal_df["ctr_march"].notna()].copy()

ctr_signal_df["ctr_bucket"] = pd.qcut(
    ctr_signal_df["ctr_march"],
    q=5,
    duplicates="drop"
)

ctr_audit = (
    ctr_signal_df
    .groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_hash_id", "count"),
        decline_rate=("declined_next_month", "mean"),
        median_ctr=("ctr_march", "median"),
        median_position=("avg_position_march", "median"),
        median_impressions=("impressions_march", "median")
    )
    .reset_index()
)

print("Signal 2 verdict: CONFIRMED")
print("n:", len(ctr_signal_df))
display(ctr_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2 verdict: CONFIRMED
n: 100893


,ctr_bucket,n,decline_rate,median_ctr,median_position,median_impressions
0,"(-0.001, 0.0494]",40357,0.588200,0.000000,13.958635,314.0
1,"(0.0494, 0.203]",20183,0.573403,0.124844,7.517361,2081.0
2,"(0.203, 0.441]",20458,0.462362,0.303951,7.013491,1591.0
3,"(0.441, 15.584]",19895,0.360593,0.701262,6.557843,838.0


Signal 1 verdict: MIXED. Staleness is a real FlyRank-style refresh signal, but in this March-to-April slice it is not clean by itself. The 181-365 day bucket has a high decline rate, but it has only 15 rows, while the large 31-90 day bucket has a lower decline rate. I should use staleness as one ingredient, not as the whole rule.

Signal 2 verdict: CONFIRMED. CTR weakness is linked to the CTR-fix logic. The CTR bucket table gives evidence that CTR is a real signal to check before using it in a baseline rule. I will use low CTR together with visibility and position so the rule does not chase tiny pages with no meaningful traffic.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Baseline rule: score, one reason code, and action label

os.makedirs("work/outputs", exist_ok=True)

baseline_df = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(ga4_sessions) AS sessions_march,
            SUM(ga4_engaged_sessions) AS engaged_sessions_march
        FROM {TABLES["fact_daily"]}
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    content AS (
        SELECT
            client_hash_id,
            content_hash_id,
            content_type,
            word_count,
            COALESCE(last_optimized_date, content_updated_date, content_created_date) AS last_content_touch
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
        WHERE is_published IS TRUE
          AND is_deleted IS FALSE
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        c.content_type,
        c.word_count,
        DATE_DIFF('day', c.last_content_touch, DATE '2026-03-31') AS days_since_last_touch,
        m.impressions_march,
        m.clicks_march,
        100.0 * m.clicks_march / NULLIF(m.impressions_march, 0) AS ctr_march,
        m.avg_position_march,
        m.sessions_march,
        100.0 * m.engaged_sessions_march / NULLIF(m.sessions_march, 0) AS engagement_rate_march
    FROM march m
    INNER JOIN content c
      ON m.client_hash_id = c.client_hash_id
     AND m.content_hash_id = c.content_hash_id
""").df()

baseline_df["baseline_score"] = (
    np.log1p(baseline_df["impressions_march"]) * 2
    + np.maximum(0, 20 - baseline_df["ctr_march"]) * 1.5
    + np.maximum(0, baseline_df["avg_position_march"] - 10) * 0.5
    + np.maximum(0, baseline_df["days_since_last_touch"] - 90) / 30
)

baseline_df["reason_code"] = "VISIBLE_LOW_CTR_POSITION_STALE"
baseline_df["action_label"] = "Review for refresh or CTR improvement"

queue = baseline_df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "content_type",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "avg_position_march",
    "days_since_last_touch",
    "engagement_rate_march",
]

queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Queue rows:", len(queue))
print("Wrote work/outputs/baseline_action_score.csv")
queue[output_cols].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue rows: 32592
Wrote work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,content_type,baseline_score,reason_code,action_label,impressions_march,clicks_march,ctr_march,avg_position_march,days_since_last_touch,engagement_rate_march
0,1,client_20259bd6705d81d4,content_c26c91a74fe92a59,keyword article,83.301626,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,140.0,1.0,0.714286,98.951069,-50,0.000000
1,2,client_3197e6291363b4db,content_b49acf92cc1c8c7e,keyword article,77.499068,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,265.0,0.0,0.000000,82.664151,22,0.000000
2,3,client_20259bd6705d81d4,content_28604ef85fd52163,keyword article,74.565048,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,115.0,1.0,0.869565,82.724432,-50,0.000000
3,4,client_ff644d8251367cbb,content_28c38e118368cd80,keyword article,73.649108,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,371.0,1.0,0.269542,74.431267,-50,0.000000
4,5,client_20259bd6705d81d4,content_9adb7795bd51a636,keyword article,73.038489,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,146.0,0.0,0.000000,76.115247,-50,0.000000
5,6,client_23a62021009f63c4,content_1ff3c48911f11e70,keyword article,72.502440,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,8581.0,0.0,0.000000,58.775190,34,0.000000
6,7,client_23a62021009f63c4,content_3ae8676b03a4401c,keyword article,72.046570,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,654.0,0.0,0.000000,68.154599,34,0.000000
7,8,client_73cda7b4e4f265ea,content_af1ad5fad6a308ff,keyword article,71.548870,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,102.0,0.0,0.000000,74.558824,-48,0.000000
8,9,client_23a62021009f63c4,content_295e883e0e86ca3c,keyword article,71.142283,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,5962.0,0.0,0.000000,57.511250,34,0.000000
9,10,client_fef1a8f436438636,content_1c47c13983830602,keyword article,71.090721,VISIBLE_LOW_CTR_POSITION_STALE,Review for refresh or CTR improvement,7290.0,6.0,0.082305,56.850773,-56,10.526316


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:

1. Action: review for refresh or CTR improvement. Why: top score from visibility, low CTR, position, and staleness. Wrong if low CTR is normal for the query.
2. Action: review for refresh or CTR improvement. Why: visible impressions with weak clicks. Wrong if tracking is incomplete.
3. Action: review for refresh or CTR improvement. Why: visible page with very low CTR. Wrong if impressions come from irrelevant queries.
4. Action: review for refresh or CTR improvement. Why: high baseline score from weak click performance. Wrong if seasonality explains it.
5. Action: review for refresh or CTR improvement. Why: impressions but no clicks. Wrong if the page is not meant to win those clicks.
6. Action: review for refresh or CTR improvement. Why: high impressions and zero clicks. Wrong if SERP features absorb the clicks.
7. Action: review for refresh or CTR improvement. Why: enough impressions to matter but no clicks. Wrong if data availability caused false zeros.
8. Action: review for refresh or CTR improvement. Why: visible impressions with zero clicks. Wrong if the page should be pruned instead.
9. Action: review for refresh or CTR improvement. Why: thousands of impressions but zero clicks. Wrong if the page has low business value.
10. Action: review for refresh or CTR improvement. Why: high impressions with very low CTR. Wrong if query intent does not match the page.
11. Action: review for refresh or CTR improvement. Why: still ranks highly under the baseline signals. Wrong if the stale date is not meaningful.
12. Action: review for refresh or CTR improvement. Why: weak CTR suggests title or snippet opportunity. Wrong if the search result already answers the query.
13. Action: review for refresh or CTR improvement. Why: position and CTR make it a review candidate. Wrong if the page is already the best answer.
14. Action: review for refresh or CTR improvement. Why: visibility makes the potential upside meaningful. Wrong if the page is outside current business priorities.
15. Action: review for refresh or CTR improvement. Why: baseline score says it deserves human review. Wrong if one noisy metric dominates the score.
16. Action: review for refresh or CTR improvement. Why: low click response with enough impressions. Wrong if the query mix is too broad.
17. Action: review for refresh or CTR improvement. Why: the rule flags it as a possible CTR-fix page. Wrong if the page targets awareness, not clicks.
18. Action: review for refresh or CTR improvement. Why: it remains near the top of the queue after sorting. Wrong if content quality is fine and demand changed.
19. Action: review for refresh or CTR improvement. Why: weak CTR or position signals push it into the top 20. Wrong if competitor or SERP changes explain the drop.
20. Action: review for refresh or CTR improvement. Why: it is the last item in the reviewed top-20 queue. Wrong if lower-ranked pages have higher client value.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A weak pick in this baseline is any page with low CTR but little realistic opportunity to win clicks. The rule does not know whether Google shows a direct answer, whether the query intent matches the page, or whether the page is strategically important. Another limitation is that the rule uses March data only and does not use future-window inputs, so it is a baseline queue, not proof that refreshing these pages will improve traffic.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.